# Crypto.Net · 05 — Polkadot dan Substrate

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./05-polkadot.en.ipynb)

Akun sr25519 dan ed25519 dengan path derivasi Substrate, alamat SS58, tanda tangan pesan, SCALE, dan extrinsic yang dibangun dari metadata runtime live dan fee-nya dihitung oleh chain.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Polkadot;

$"sr25519 needs the Rust core — loaded: {NativeLoader.IsNativeAvailable}"

## Akun pengembangan

`//Alice` diturunkan dari frasa pengembangan publik Substrate — akun yang sama yang diberi saldo oleh setiap node lokal.

In [ ]:
var alice   = PolkadotAccount.FromSuri("//Alice", chain: PolkadotChain.Westend);
var aliceEd = PolkadotAccount.FromSuri("//Alice", SignatureScheme.Ed25519, PolkadotChain.Westend);

new[] { PolkadotChain.Polkadot, PolkadotChain.Kusama, PolkadotChain.Westend }
    .Select(c => new { Network = c.Name, c.Ss58Prefix, Sr25519 = alice.AddressFor(c), Ed25519 = aliceEd.AddressFor(c) })

## Akun Anda

Path kosong adalah akun root yang ditampilkan Polkadot.js, Talisman dan SubWallet untuk sebuah frasa; junction `//hard` dan `/soft` menurunkan akun lainnya.

In [ ]:
var wallet = HdWallet.Generate();
var root   = wallet.GetPolkadotAccount("", PolkadotChain.Polkadot);
var child  = wallet.GetPolkadotAccount("//polkadot//0", PolkadotChain.Polkadot);
var (prefix, publicKey) = Ss58Address.Decode(root.Address.Value);
Console.WriteLine($"root  {root.Address}  (prefix {prefix}, key {HexUtil.Encode(publicKey)})");
Console.WriteLine($"child {child.Address}  ({child.DerivationPath})");

## Menandatangani pesan

`SignMessage` membungkus teks dengan `<Bytes>…</Bytes>` seperti `signRaw` di ekstensi Polkadot.js. Tanda tangan sr25519 bersifat acak sehingga berbeda tiap kali, tetapi selalu terverifikasi.

In [ ]:
byte[] sig = root.SignMessage("Hello Polkadot");
Console.WriteLine(HexUtil.Encode(sig));
Console.WriteLine($"valid: {PolkadotAccount.VerifyMessage("Hello Polkadot", sig, root.PublicKey.Span)}");
Console.WriteLine($"tampered: {PolkadotAccount.VerifyMessage("Hello Kusama", sig, root.PublicKey.Span)}");

## Codec SCALE

In [ ]:
var encoded = new ScaleWriter().Compact(1_000_000_000_000).U32(42).String("Crypto.Net").ToArray();
{
    // ScaleReader is a ref struct, so it lives inside a block rather than as a notebook-level variable.
    var reader = new ScaleReader(encoded);
    Console.WriteLine($"{HexUtil.Encode(encoded)} → {reader.Compact()}, {reader.U32()}, {reader.String()}");
}

## Metadata runtime live (Westend Asset Hub)

Indeks pallet, indeks call dan daftar signed extension dibaca dari chain, sehingga extrinsic tetap berfungsi setelah upgrade runtime.

In [ ]:
var rpc = new PolkadotRpcClient(PolkadotChain.WestendAssetHub);
var (spec, txVersion, specName) = await rpc.GetRuntimeVersionAsync();
var metadata = await rpc.GetMetadataAsync();
var (pallet, callIndex) = metadata.GetCallIndex("Balances", "transfer_keep_alive");

Console.WriteLine($"{await rpc.GetChainNameAsync()} · {specName} v{spec} · metadata V{metadata.Version} · {metadata.Pallets.Count} pallets");
Console.WriteLine($"Balances.transfer_keep_alive = ({pallet}, {callIndex})");
Console.WriteLine($"signed extensions: {string.Join(", ", metadata.SignedExtensions.Select(e => e.Identifier))}");
var aliceInfo = await rpc.GetAccountInfoAsync(alice.AddressFor(PolkadotChain.WestendAssetHub));
Console.WriteLine($"//Alice on Westend Asset Hub: nonce {aliceInfo.Nonce}, free {new Amount(aliceInfo.Free, 12)} WND");

## Extrinsic bertanda tangan, dihargai oleh chain

Runtime harus men-decode extrinsic untuk menghitung fee-nya, sehingga estimasi fee membuktikan encoding-nya valid. Tidak ada yang dikirim.

In [ ]:
var ctx = await rpc.GetExtrinsicContextAsync(alice.Address.Value);
byte[] call = Extrinsic.TransferKeepAlive(metadata, root.AddressFor(PolkadotChain.WestendAssetHub), 1_000_000_000_000);
byte[] extrinsic = Extrinsic.Sign(metadata, call, alice, ctx);

Console.WriteLine($"{extrinsic.Length} bytes, hash {Extrinsic.Hash(extrinsic)}");
Console.WriteLine($"mortal for {ctx.MortalPeriod} blocks from #{ctx.BlockNumber:N0}, nonce {ctx.Nonce}");
Console.WriteLine($"fee quoted by the runtime: {await rpc.EstimateFeeAsync(extrinsic)} WND");

## Mengirim sungguhan

```csharp
TxHash hash = await rpc.TransferAsync(root, "5Grw…", Amount.Parse("0.5", 12));
TxHash remark = await rpc.SubmitCallAsync(root, Extrinsic.Remark(metadata, "hello"u8));
```

In [ ]:
foreach (IDisposable d in new IDisposable[] { alice, aliceEd, root, child, rpc, wallet }) d.Dispose();
"disposed"